### Read the sales_details data from the Bronze layer, apply the required cleaning and standardization steps, then save the final cleaned output to the Silver layer as a Delta table.

# Initialization


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

# Display the raw Bronze data first 

In [0]:
# Display the raw Bronze data first to understand the file structure
# and identify the data quality issues that need cleaning in the Silver layer.
df.display()

- ============================================================

Data quality issues found in this file:
 1. Some string columns may contain leading or trailing spaces.
 2. Date columns may contain invalid values such as 0 or values
    that do not follow the yyyyMMdd format.
 3. The sls_price column may contain null, zero, or negative values.
 4. Missing or invalid sls_price values can be derived from
   sls_sales / sls_quantity when quantity is not zero.
  5.Column names need to be renamed to cleaner business-friendly names before saving the final Silver table.
  -
============================================================

# Silver Transformations



# Trimming



In [0]:
# Trimming string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Cleaning Dates

In [0]:
# Cleaning date columns
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt").cast("string")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt").cast("string")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt").cast("string")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)


# Sales and Price Corrections

In [0]:
# Validation checks before fixing price
df.select(
    F.count(F.when(col("sls_price").isNull(), True)).alias("null_price_count"),
    F.count(F.when(col("sls_price") < 0, True)).alias("negative_price_count"),
    F.count(F.when(col("sls_price") == 0, True)).alias("zero_price_count")
).display()

In [0]:
# Fixing missing or invalid price values
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                F.abs(col("sls_sales")) / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

In [0]:
# Validation checks after fixing price
df.select(
    F.count(F.when(col("sls_price").isNull(), True)).alias("null_price_count"),
    F.count(F.when(col("sls_price") < 0, True)).alias("negative_price_count"),
    F.count(F.when(col("sls_price") == 0, True)).alias("zero_price_count")
).display()

# check duplicate record on bussiness keys in this file

In [0]:
# Check for duplicate records based on the business key
# to make sure the sales rows are unique before saving to Silver.
# Count duplicate business keys
df.groupBy("sls_ord_num", "sls_prd_key", "sls_cust_id") \
  .count() \
  .filter(col("count") > 1) \
  .count()

#rename coulmns

In [0]:

# Renaming columns to cleaner business names
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


#quick validation for df

In [0]:
# Display the cleaned dataframe for a quick validation check
df.display()


#writing silver table

In [0]:
# Writing Silver table
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")


#check table created or not 

In [0]:
%sql
-- Preview the final Silver table
SELECT * FROM workspace.silver.crm_sales LIMIT 10